In [ ]:
import os
import cv2, tqdm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
from keras.models import Sequential
from keras.utils import to_categorical
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [ ]:
image_dir = 'input/train'
train_folders = os.listdir(image_dir)
Sx = 100
Sy = 100
epocas = 20
train_size = 100
test_size = 20

In [ ]:
print(f"Min files in a folder: {min([len([f for f in os.listdir(os.path.join(image_dir, fldr)) if os.path.isfile(os.path.join(image_dir, fldr, f))]) for fldr in train_folders])}")
total_images = sum(
    len([f for f in os.listdir(os.path.join(image_dir, fldr)) if os.path.isfile(os.path.join(image_dir, fldr, f))])
    for fldr in train_folders
)
print(f"Total images: {total_images}")

In [ ]:
def exploreData (folders) :
    plt.figure(figsize=(10,10))
    for i,fol in tqdm.tqdm(enumerate(folders[:25])) :
        plt.subplot(5,5 ,i+1)
        image = os.listdir(os.path.join(image_dir,fol))[0]
        imgPath = os.path.join(image_dir,fol,image)
        img = cv2.imread(imgPath)
        imgRGB = cv2.cvtColor(img , cv2.COLOR_BGR2RGB)
        imgRGB = cv2.resize(imgRGB, (Sx,Sy))
        plt.title(fol)
        plt.imshow(imgRGB)
    plt.show()
exploreData(train_folders)

In [ ]:
x_test_RGB = []
y_test_RGB = []
x_valid_RGB = []
y_valid_RGB = []
x_train_RGB = []
y_train_RGB = []

random.seed(42)

for folder in tqdm.tqdm(train_folders):
    folder_path = os.path.join(image_dir, folder)
    images_in_folder = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    random.shuffle(images_in_folder)

    valid_images = images_in_folder[-test_size:]
    test_images = images_in_folder[:test_size]
    train_images = images_in_folder[:train_size]

    for img_path in test_images:
        img = cv2.imread(img_path)
        imgRGB = cv2.cvtColor(img , cv2.COLOR_BGR2RGB)
        imgRGB = cv2.resize(imgRGB, (Sx,Sy))
        x_test_RGB.append(imgRGB)
        y_test_RGB.append(folder)

    for img_path in valid_images:
        img = cv2.imread(img_path)
        imgRGB = cv2.cvtColor(img , cv2.COLOR_BGR2RGB)
        imgRGB = cv2.resize(imgRGB, (Sx,Sy))
        x_valid_RGB.append(imgRGB)
        y_valid_RGB.append(folder)

    for img_path in train_images:
        img = cv2.imread(img_path)
        imgRGB = cv2.cvtColor(img , cv2.COLOR_BGR2RGB)
        imgRGB = cv2.resize(imgRGB, (Sx,Sy))
        x_train_RGB.append(imgRGB)
        y_train_RGB.append(folder)

x_test_RGB = np.array(x_test_RGB)
x_valid_RGB = np.array(x_valid_RGB)
x_train_RGB = np.array(x_train_RGB)

x_test_RGB = x_test_RGB.astype('float32') / 255.0
x_valid_RGB = x_valid_RGB.astype('float32') / 255.0
x_train_RGB = x_train_RGB.astype('float32') / 255.0

x_test_RGB = x_test_RGB.reshape(-1,Sx,Sy,3)
x_valid_RGB = x_valid_RGB.reshape(-1,Sx,Sy,3)
x_train_RGB = x_train_RGB.reshape(-1,Sx,Sy,3)

df = pd.read_csv('input/cards.csv')
category_labels = df.labels.value_counts().index
categories = {}
for index,cat in enumerate(category_labels) :
  categories[cat]=index

y_train_RGB = [categories[i] for i in y_train_RGB]
y_valid_RGB = [categories[i] for i in y_valid_RGB]
y_test_RGB = [categories[i] for i in y_test_RGB]

y_train_RGB = to_categorical(y_train_RGB)
y_valid_RGB = to_categorical(y_valid_RGB)
y_test_RGB = to_categorical(y_test_RGB)